# Silver Layer — Stock Market Data

**Purpose:** Standardize historical market data (JPM, GS, etc.) to enable cross-signal correlation with IBM AML transactions.

In [10]:
# ============================================================
# FinPulse — Silver Layer: IBM AML Transactions
# Purpose: Clean, enrich IBM AML transaction data
# Storage: Apache Iceberg v2 Hadoop catalog
# Author: Amanjot Kaur
# ============================================================
import os
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True)

def _find_project_root(marker='pyproject.toml') -> Path:
    curr = Path(os.getcwd()).resolve()
    for cand in [curr] + list(curr.parents):
        if (cand / marker).exists(): return cand
    return curr

PROJECT_ROOT = _find_project_root()

# ── Clear bad SPARK_HOME ──────────────────────────────────────
if 'SPARK_HOME' in os.environ and not Path(os.environ['SPARK_HOME']).is_dir():
    del os.environ['SPARK_HOME']

# ── Environment ───────────────────────────────────────────────
os.environ['JAVA_HOME']             = r'C:\Program Files\Microsoft\jdk-11.0.30.7-hotspot'
os.environ['HADOOP_HOME']           = str(PROJECT_ROOT / 'hadoop')
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

for p in [
    Path(os.environ['JAVA_HOME']) / 'bin',
    Path(os.environ['HADOOP_HOME']) / 'bin',
]:
    if p.is_dir() and str(p) not in os.environ['PATH']:
        os.environ['PATH'] = str(p) + os.pathsep + os.environ['PATH']

# ── Paths ─────────────────────────────────────────────────────
BRONZE_STOCKS     = str(PROJECT_ROOT / 'data' / 'bronze' / 'stocks')
ICEBERG_WAREHOUSE = os.environ.get(
    'ICEBERG_WAREHOUSE',
    str(PROJECT_ROOT / 'data' / 'silver' / 'iceberg_warehouse')
)
ICEBERG_JAR = str(PROJECT_ROOT / 'jars' /
    'iceberg-spark-runtime-3.5_2.12-1.4.3.jar')

PIPELINE_VERSION    = 'v2.0'
INGESTION_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Project root     : {PROJECT_ROOT}')
print(f'Bronze path      : {BRONZE_STOCKS}')
print(f'Iceberg warehouse: {ICEBERG_WAREHOUSE}')
print(f'JAR exists       : {Path(ICEBERG_JAR).exists()}')

# ── Spark + Iceberg ───────────────────────────────────────────
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType, DoubleType, TimestampType

TEMP_DIR = Path.home() / 'FinPulse_Spark_Temp'
os.makedirs(TEMP_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('FinPulse-Silver-Stocks')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    # ── Windows critical fixes ────────────────────────────────
    .config('spark.driver.extraJavaOptions',
            f'-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false'
            f' -Djava.io.tmpdir="{str(TEMP_DIR).replace(chr(92), "/")}"')
    .config('spark.executor.extraJavaOptions',
            '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false')
    # Replaced RawLocalFileSystem with LocalFileSystem for Iceberg compatibility
    .config('spark.hadoop.fs.file.impl',
            'org.apache.hadoop.fs.LocalFileSystem')
    .config('spark.hadoop.fs.file.impl.disable.cache', 'true')
    .config('spark.hadoop.fs.verify.checksum', 'false')
    # ── Iceberg ───────────────────────────────────────────────
    .config('spark.jars', ICEBERG_JAR)
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local',
            'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', ICEBERG_WAREHOUSE)
    .config('spark.sql.defaultCatalog', 'local')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'\n✅ Spark + Iceberg ready | version: {spark.version}')


Project root     : C:\FinPulse Project
Bronze path      : C:\FinPulse Project\data\bronze\stocks
Iceberg warehouse: C:\FinPulse Project\data\silver\iceberg_warehouse
JAR exists       : True

✅ Spark + Iceberg ready | version: 3.5.8


In [11]:
# Find latest partition
partitions = [d for d in os.listdir(BRONZE_STOCKS) if d.startswith("date=")]
latest_partition = sorted(partitions)[-1]
partition_dir = Path(BRONZE_STOCKS) / latest_partition

# Handle both 'stocks_raw.csv' and timestamp-versioned files
csv_files = [f for f in os.listdir(partition_dir) if f.endswith(".csv")]
target_file = partition_dir / (csv_files[-1] if csv_files else "stocks_raw.csv")

print(f"Loading: {target_file}")
df_raw = spark.read.csv(str(target_file), header=True, inferSchema=True)
print(f"Raw records: {df_raw.count():,}")

Loading: C:\FinPulse Project\data\bronze\stocks\date=2026-04-20\stocks_raw.csv
Raw records: 175


In [12]:
def apply_stocks_transformations(df):
    print("Applying Stocks Silver transformations...")
    
    # Step 1 — Standardize column names to lowercase snake_case
    for col in df.columns:
        new_col = col.lower().replace(" ", "_")
        df = df.withColumnRenamed(col, new_col)
    print("Step 1 — Column names standardised")

    # Step 2 — Cast date properly
    df = df.withColumn("date", F.to_date(F.col("date")))
    print("Step 2 — Date cast to DateType")

    # Step 3 — Daily return %: (close-open)/open*100
    df = df.withColumn(
        "daily_return",
        F.round((F.col("close") - F.col("open")) / F.col("open") * 100, 4)
    )
    print("Step 3 — daily_return calculated")

    # Step 4 — Price change from previous day using lag()
    # Window partitioned by ticker ordered by date
    # lag(1) = previous trading day close for same ticker
    window_spec = Window.partitionBy("ticker").orderBy("date")
    df = df.withColumn(
        "price_change",
        F.round(F.col("close") - F.lag("close", 1).over(window_spec), 4)
    )
    print("Step 4 — price_change via lag() window function")

    # Step 5 — Is positive day (close > open)
    df = df.withColumn(
        "is_positive_day",
        F.when(F.col("close") > F.col("open"), True).otherwise(False)
    )
    positive = df.filter(F.col("is_positive_day")).count()
    total = df.count()
    print(f"Step 5 — is_positive_day: {positive:,}/{total:,} positive days")

    # Step 6 — Price range (high - low) = daily volatility
    df = df.withColumn(
        "price_range",
        F.round(F.col("high") - F.col("low"), 4)
    )
    print("Step 6 — price_range = high-low daily volatility")

    # Step 7 — 7 day moving average
    window_7d = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)
    df = df.withColumn(
        "moving_avg_7d",
        F.round(F.avg("close").over(window_7d), 4)
    )
    print("Step 7 — 7 day moving average calculated")

    # Step 8 — 30 day moving average
    window_30d = Window.partitionBy("ticker").orderBy("date").rowsBetween(-29, 0)
    df = df.withColumn(
        "moving_avg_30d",
        F.round(F.avg("close").over(window_30d), 4)
    )
    print("Step 8 — 30 day moving average calculated")

    # Step 9 — Pipeline metadata
    df = df.withColumn("silver_processed_at", F.current_timestamp())
    df = df.withColumn("pipeline_version", F.lit("v2.0"))
    print("Step 9 — Pipeline metadata added")

    print(f"\n✅ Transformations complete")
    print(f"Final records: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")
    return df

silver_df = apply_stocks_transformations(df_raw)
silver_df.printSchema()


Applying Stocks Silver transformations...
Step 1 — Column names standardised
Step 2 — Date cast to DateType
Step 3 — daily_return calculated
Step 4 — price_change via lag() window function
Step 5 — is_positive_day: 83/175 positive days
Step 6 — price_range = high-low daily volatility
Step 7 — 7 day moving average calculated
Step 8 — 30 day moving average calculated
Step 9 — Pipeline metadata added

✅ Transformations complete
Final records: 175
Final columns: 19
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- dividends: double (nullable = true)
 |-- stock_splits: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- data_source: string (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |--

In [ ]:
# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("STOCKS SILVER VALIDATION")
print("=" * 60)

checks_passed = 0

# Check 1 - No negative close
neg_close = silver_df.filter(F.col("close") <= 0).count()
if neg_close == 0:
    print("✅ Check 1 PASSED — No negative close prices")
    checks_passed += 1
else:
    print(f"❌ Check 1 FAILED — {neg_close} negative close prices")

# Check 2 - All tickers present
tickers = [r[0] for r in silver_df.select("ticker").distinct().collect()]
print(f"✅ Check 2 — Tickers present: {sorted(tickers)}")
checks_passed += 1

# Check 3 - No null close
null_close = silver_df.filter(F.col("close").isNull()).count()
if null_close == 0:
    print("✅ Check 3 PASSED — No null close prices")
    checks_passed += 1
else:
    print(f"❌ Check 3 FAILED — {null_close} null closes")

# Ticker summary
print("\n── Ticker Summary ──")
silver_df.groupBy("ticker").agg(
    F.count("date").alias("trading_days"),
    F.round(F.avg("close"), 2).alias("avg_close"),
    F.round(F.avg("daily_return"), 4).alias("avg_daily_return_pct"),
    F.round(F.avg("price_range"), 2).alias("avg_daily_range")
).orderBy("ticker").show()

print(f"Validation complete — {checks_passed}/3 passed")

# ── Write to Iceberg ──────────────────────────────────────────
print("\n" + "=" * 60)
print("WRITING STOCKS SILVER — Apache Iceberg")
print("=" * 60)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.finpulse")
print("✅ Namespace local.finpulse ready")

silver_df.writeTo("local.finpulse.silver_stocks") \
    .tableProperty("format-version", "2") \
    .tableProperty("write.parquet.compression-codec", "snappy") \
    .createOrReplace()

print("✅ Iceberg write complete")

# ── Verify via file system (avoids RawLocalFileSystem read-back issue on Windows)
from pathlib import Path
iceberg_table_path = Path(ICEBERG_WAREHOUSE) / "finpulse" / "silver_stocks" / "data"

if iceberg_table_path.exists():
    parquet_files = list(iceberg_table_path.rglob("*.parquet"))
    total_size = sum(f.stat().st_size for f in parquet_files) / (1024 * 1024)
    print(f"✅ Iceberg table verified on disk:")
    print(f"   Parquet files : {len(parquet_files)}")
    print(f"   Total size    : {total_size:.1f} MB")
    print(f"   Location      : {iceberg_table_path}")
else:
    print("❌ Iceberg data folder not found — write may have failed")

# ── Summary ───────────────────────────────────────────────────
print("=" * 60)
print("FINPULSE SILVER STOCKS — COMPLETE SUMMARY")
print("=" * 60)
print(f"Source          : Yahoo Finance (Aug-Sep 2022)")
print(f"Period          : 2022-08-15 to 2022-09-30")
print(f"Tickers         : AAPL, GOOGL, MSFT, JPM, GS")
print(f"Columns         : {len(silver_df.columns)}")
print(f"New columns     : daily_return, price_change, is_positive_day,")
print(f"                  price_range, moving_avg_7d, moving_avg_30d")
print(f"Iceberg table   : local.finpulse.silver_stocks")
print(f"Storage format  : Apache Iceberg v2, Snappy compression")
print(f"Purpose         : Cross-signal join with IBM AML transactions on date")
print("=" * 60)

spark.stop()
print("\n✅ Spark stopped — Silver stocks complete")

STOCKS SILVER VALIDATION
✅ Check 1 PASSED — No negative close prices
✅ Check 2 — Tickers present: ['AAPL', 'GOOGL', 'GS', 'JPM', 'MSFT']
✅ Check 3 PASSED — No null close prices

── Ticker Summary ──
+------+------------+---------+--------------------+---------------+
|ticker|trading_days|avg_close|avg_daily_return_pct|avg_daily_range|
+------+------------+---------+--------------------+---------------+
|  AAPL|          35|   142.52|             -0.3761|           2.88|
| GOOGL|          35|   103.88|             -0.3564|           2.05|
|    GS|          35|   338.28|              0.1855|           6.89|
|   JPM|          35|   117.88|               0.097|           1.98|
|  MSFT|          35|   266.66|              0.2444|           5.32|
+------+------------+---------+--------------------+---------------+

Validation complete — 3/3 passed

WRITING STOCKS SILVER — Apache Iceberg
✅ Namespace local.finpulse ready
✅ Records in Iceberg table: 175

── Sample Silver Stocks Data ──


UnsupportedOperationException: Not implemented by the RawLocalFileSystem FileSystem implementation